In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
!pip install cvzone
!pip install yt-dlp
!pip install ultralytics
!pip install sort-tracker-py
!pip install pydub moviepy

  Preparing metadata (setup.py) ... done
  Created wheel for cvzone: filename=cvzone-1.6.1-py3-none-any.whl size=26297 sha256=eba466fffb1b14c576e2c6b7defb6a679a0d22640bfdcefd3766123f2e8f2d42
  Stored in directory: /root/.cache/pip/wheels/2c/9f/b3/92e945ac4a71bf727a92463f38155cc5a4fa49c5010b38ec4c
Successfully built cvzone
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 171.3/171.3 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 61.3 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.1/164.1 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.4/194.4 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 73.0 MB/s eta 0:00:00
  Attempting uninstall: websockets
    Found existing installation: websockets 12.0
    Uninstalling websockets-12.0:
      Successfully uninstalled websockets-12.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 871.0/871.0 kB 18.6 MB/s eta 0:00:0000:01
     ━━━━━━━━━━━━━━━━━

In [2]:
import numpy as np
from ultralytics import YOLO
import cv2
import cvzone
import math
from sort_tracker import *
from pydub import AudioSegment
from pydub.playback import play
from moviepy.editor import VideoFileClip, AudioFileClip

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [3]:

cap = cv2.VideoCapture("/kaggle/input/iiuc-1080p/iiuc_rail_gate.mp4")


if not cap.isOpened():
    print("Error: Could not open video.")
    exit()


# Load the alarm sound
alarm_sound = "/content/MV27TES-alarm.mp3"



model = YOLO("yolo11l.pt")

classNames = ["person", "bicycle", "car", "motorbike", "aeroplane", "bus", "train", "truck", "boat",
              "traffic light", "fire hydrant", "stop sign", "parking meter", "bench", "bird", "cat",
              "dog", "horse", "sheep", "cow", "elephant", "bear", "zebra", "giraffe", "backpack", "umbrella",
              "handbag", "tie", "s  uitcase", "frisbee", "skis", "snowboard", "sports ball", "kite", "baseball bat",
              "baseball glove", "skateboard", "surfboard", "tennis racket", "bottle", "wine glass", "cup",
              "fork", "knife", "spoon", "bowl", "banana", "apple", "sandwich", "orange", "broccoli",
              "carrot", "hot dog", "pizza", "donut", "cake", "chair", "sofa", "pottedplant", "bed",
              "diningtable", "toilet", "tvmonitor", "laptop", "mouse", "remote", "keyboard", "cell phone",
              "microwave", "oven", "toaster", "sink", "refrigerator", "book", "clock", "vase", "scissors",
              "teddy bear", "hair drier", "toothbrush"
              ]

mask = cv2.imread("/kaggle/input/iiuc-1080p/mask.png")

# tracker
tracker = Sort(max_age=20,min_hits=3,iou_threshold=0.3)

crossline = [878, 359, 878, 722]

tot_bus = set()
# print(model)



width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)
     
# Define the codec and create VideoWriter object
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter('iiuc2.mp4', fourcc, fps, (width,height))

if mask.shape[:2] != (height, width):
    print("Error: Mask dimensions do not match the video frame dimensions.")
    exit()
while cap.isOpened():

    success, img = cap.read()
    
#     img = cv2.resize(img,(1280,720))
#     mask = cv2.resize(mask,(1280,720))
    if not success:
        break
#     if img==None:
#         break
        
    if mask.shape[:2] != img.shape[:2]:
        print("Error: Mask dimensions do not match the video frame dimensions.")
        exit()
    img_region = cv2.bitwise_and(img,mask)# joining frame and mask
    if not success:
        print("Error: Could not read frame.")
        break
    detections = np.empty((0,5))

    results = model(img_region, stream=True)
    for r in results:
        #bounding boxex
        boxes = r.boxes
        for box in boxes:
            x1, y1, x2, y2 = box.xyxy[0]
            x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
            w, h = x2 - x1, y2 - y1




            conf = math.ceil(box.conf[0] * 100) / 100
            class_name = int(box.cls[0])
            # print(class_name)
            if classNames[class_name] == 'train':
                cvzone.cornerRect(img, (x1, y1, w, h),l=9,rt=2)
                cvzone.putTextRect(img, f"{classNames[class_name]}", (max(0, x1), max(35, y1)),
                                 scale=1,thickness=1,offset=1)
#                 play(AudioSegment.from_mp3(alarm_sound))  # Play the alarm sound

            if classNames[class_name] == 'bus':
                # cvzone.cornerRect(img, (x1, y1, w, h),l=9,rt=5)
                # cvzone.putTextRect(img, f"{classNames[class_name]}{conf}", (max(0, x1), max(35, y1)),
                #                  scale=1,thickness=1,offset=4)

                currentArray = np.array([x1,y1,x2,y2,conf])
                detections = np.vstack((detections,currentArray))

    resultsTracker = tracker.update(detections)

    cv2.line(img,(crossline[0],crossline[1]),(crossline[2],crossline[3]),(0,0,255),1)

    for result in resultsTracker:
        x1,y1,x2,y2,conf,Id = result
        print(result)
        x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
        w, h = x2 - x1, y2 - y1
        cvzone.cornerRect(img, (x1, y1, w, h),l=9,rt=2,colorR=(255,0,0))
        # cvzone.putTextRect(img, f"Tracking {Id}", (max(0, x1), max(35, y1)),
        #                     scale=1,thickness=1,offset=4,colorB=(255,0,0))

        '''coming line'''
        # center of each object
        cx,cy = x1+w//2,y1+h//2
        cv2.circle(img,(cx,cy),3,(0,0,255),cv2.FILLED)

        if crossline[2]-20<cx<crossline[2]+20 and crossline[1]<cy<crossline[3]:
            tot_bus.add(Id)
            cv2.line(img,(crossline[0],crossline[1]),(crossline[2],crossline[3]),(0,255,0),1)
        # print(coming_count)
    cvzone.putTextRect(img,f"IIUC Bus:{len(tot_bus)}",(0,50),colorR=(0,127,0))

        # cv2.putText(img,str(len(coming_count)),(255,100),cv2.FONT_HERSHEY_SIMPLEX,3,(255,255,0))


#     cv2.waitKey(0)


    # cv2.imshow("IIUC Rail And Bus crossing", img)
    out.write(img)


cap.release()
out.release()
# cv2.destroyAllWindows()

# video_clip = VideoFileClip('output.mp4')
# audio_clip = AudioFileClip(alarm_sound)
# final_clip = video_clip.set_audio(audio_clip)
# final_clip.write_videofile('final_output.mp4', codec='libx264')


0: 384x640 1 train, 124.2ms
Speed: 11.0ms preprocess, 124.2ms inference, 248.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 1 train, 1 truck, 18.2ms
Speed: 2.7ms preprocess, 18.2ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 1 train, 18.3ms
Speed: 2.7ms preprocess, 18.3ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 1 train, 18.2ms
Speed: 2.0ms preprocess, 18.2ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 1 train, 22.9ms
Speed: 3.0ms preprocess, 22.9ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 1 train, 18.2ms
Speed: 2.7ms preprocess, 18.2ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 persons, 1 train, 18.2ms
Speed: 3.3ms preprocess, 18.2ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 persons, 1 train, 18.6ms
Speed: 2.7m